In [1]:
import math

# Simpson's rule 
def simpson_integral(f, a, b, n):
   
    if n % 2 == 1:
        raise ValueError("n must be even for Simpson's rule")

    h = (b - a) / n
    total = f(a) + f(b)

    # odd indices: 4 * f(...)
    for i in range(1, n, 2):
        x = a + i * h
        total += 4 * f(x)

    # even indices: 2 * f(...)
    for i in range(2, n, 2):
        x = a + i * h
        total += 2 * f(x)

    return (h / 3) * total



# keep iterating until tolerance
# 
def integral_until_converged(f, a, b, tol=1e-12):
    
    n = 4  # start with a small even number
    I_old = simpson_integral(f, a, b, n)

    while True:
        n = n * 2
        I_new = simpson_integral(f, a, b, n)

        if abs(I_new - I_old) < tol:
            return I_new, n

        I_old = I_new


# Standard normal CDF using Simpson integral
def N_simpson(x, tol=1e-12):
    """
    Standard normal CDF N(x) computed as:
      N(x) = 1/2 + (1/sqrt(2pi)) * integral_0^x exp(-t^2/2) dt  (x>=0)
    and symmetry for x<0.
    """
    if x == 0:
        return 0.5

    if x < 0:
        return 1.0 - N_simpson(-x, tol)

    def f(t):
        return math.exp(-t*t / 2)

    I, n_used = integral_until_converged(f, 0.0, x, tol)
    return 0.5 + (1.0 / math.sqrt(2.0 * math.pi)) * I


# Black–Scholes call price (with dividend yield q)
def black_scholes_call(S0, K, r, q, sigma, T, tol=1e-12):
    sqrtT = math.sqrt(T)

    d1 = (math.log(S0 / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT

    Nd1 = N_simpson(d1, tol)
    Nd2 = N_simpson(d2, tol)

    C = S0 * math.exp(-q * T) * Nd1 - K * math.exp(-r * T) * Nd2
    return C, d1, d2, Nd1, Nd2



S0 = 40
K = 40
T = 0.25        
sigma = 0.20
q = 0.01
r = 0.05

C, d1, d2, Nd1, Nd2 = black_scholes_call(S0, K, r, q, sigma, T, tol=1e-12)

print("d1 =", d1)
print("d2 =", d2)
print("N(d1) =", Nd1)
print("N(d2) =", Nd2)
print("Call price C =", C)

d1 = 0.15
d2 = 0.04999999999999999
N(d1) = 0.5596176923702444
N(d2) = 0.5199388058383745
Call price C = 1.7896149290761159
